# LLM Lab 10: Alignment Experiment (DPO)
University of Gwadar | BS IT 8th Semester | Base model: `Qwen/Qwen2.5-0.5B` | Dataset: `trl-lib/ultrafeedback_binarized`

**How to use this notebook**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells **one by one, from top to bottom**. Read the short note above each cell first.
3. Cells marked **YOU DO THIS** need your own writing or scoring.

**Time:** about 1 to 1.5 hours in total (most of it is training). Keep this browser tab open the whole time.


## STEP 1: Install the libraries
If Colab shows a message asking you to **restart the session**, click Restart, then run this cell once more and continue.


In [ ]:
!pip install -q -U trl peft transformers datasets accelerate huggingface_hub gradio "pandas==2.2.3"
!pip install -q "pyarrow>=24.0.0"


## STEP 2: Log in to Hugging Face
Run the cell. It shows a **link and a short code**. Open the link, enter the code and approve. No token needs to be typed.

If it asks you to paste a token instead, create a token with **Write** access at huggingface.co/settings/tokens and paste it in the box. Never type a token into a notebook cell that you save or share.


In [ ]:
from huggingface_hub import login, whoami
login()
HF_USER = whoami()["name"]
print("Logged in as:", HF_USER)


## STEP 3: Experiment design and the base model before DPO
**Lab requirements covered:** your own prompts (6 prompts, 5 task types) and testing the base model before alignment.

Every model in this lab uses exactly the same generation settings (greedy decoding, repetition penalty 1.1, up to 200 new tokens), so any difference between models comes from training and not from randomness.


In [ ]:
import torch, gc, json
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-0.5B"

prompts = [
 "What is PPO (Proximal Policy Optimization) in reinforcement learning? Explain simply.",
 "What is the difference between PPO and DPO?",
 "I have a job interview in 2 days and I feel stressed. What should I do?",
 "Write a short, polite message to my class teacher asking for an extension on my assignment deadline.",
 "If a car travels 5 km in 2.5 hours, what is its speed? Show steps.",
 "Why did the Khan of Kalat give Gwadar to Oman in 1783?",
]
task_types = ["Explanation", "Explanation", "Advice", "Professional writing", "Reasoning", "Question answering"]

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).cuda().eval()
    return tok, model

def generate(tok, model, prompt, max_new_tokens=200):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to("cuda")
    stop = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=1.1, eos_token_id=stop,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

# ---- test the BASE model before alignment ----
tok_b, base_model = load_model(BASE)
base_out = [generate(tok_b, base_model, p) for p in prompts]
for t, p, r in zip(task_types, prompts, base_out):
    print(f"[{t}] PROMPT: {p}\nBASE: {r}\n" + "-" * 70)
json.dump(dict(zip(prompts, base_out)), open("base_outputs.json", "w"), indent=2)

del base_model
gc.collect(); torch.cuda.empty_cache()


**YOU DO THIS:** In a text cell below, write 3 to 4 sentences about what the base model did well and badly (for example: does it stop cleanly, repeat itself, follow the instruction, invent facts?). This is your "before" evidence.

## STEP 4: Inspect the preference data
**Lab requirement:** inspect chosen and rejected responses and explain why the chosen one was preferred.


In [ ]:
from datasets import load_dataset

ds = load_dataset("trl-lib/ultrafeedback_binarized")
print(ds)

for i in range(5):
    ex = ds["train"][i]
    print(f"\n===== EXAMPLE {i + 1} =====")
    print("PROMPT  :", ex["chosen"][0]["content"][:500])
    print("\nCHOSEN  :", ex["chosen"][-1]["content"][:600])
    print("\nREJECTED:", ex["rejected"][-1]["content"][:600])
    print("\nSCORES  : chosen =", ex["score_chosen"], "| rejected =", ex["score_rejected"])

sub = ds["train"].shuffle(seed=1).select(range(1000))
cl = [len(x[-1]["content"]) for x in sub["chosen"]]
rl = [len(x[-1]["content"]) for x in sub["rejected"]]
print("\n===== STATISTICS (1000 random pairs) =====")
print("Average length, chosen   :", round(np.mean(cl)), "characters")
print("Average length, rejected :", round(np.mean(rl)), "characters")
print("Chosen is longer in      :", round(np.mean([c > r for c, r in zip(cl, rl)]) * 100, 1), "% of pairs")
print("Average score, chosen    :", round(np.mean(sub["score_chosen"]), 2))
print("Average score, rejected  :", round(np.mean(sub["score_rejected"]), 2))


**YOU DO THIS:** Write a small table or paragraph (in a text cell) for the 5 examples above:

| Example | Task | Why was the chosen response preferred? (clearer? more complete? follows the instruction? polite tone? no mistakes?) |
|---|---|---|

Then answer: *Do you see a repeated pattern across the chosen responses?* Use the statistics printed above as evidence.

## STEP 5: Train DPO with two beta values and publish both models
**Lab requirements covered:** align the model with DPO, beta = 0.1 as the baseline, and one additional beta (0.5).

One function trains a model and uploads it to **your** Hugging Face account. It runs twice, so you get **two model repos automatically**:
- `<your-username>/qwen2.5-0.5b-dpo-beta01`
- `<your-username>/qwen2.5-0.5b-dpo-beta05`

This is the slowest step (about 20 to 40 minutes per run). Keep the tab open.

**Settings to record in your report:** LoRA r = 16, alpha = 32, dropout 0.05, learning rate 5e-5, batch size 2 with gradient accumulation 8, 1 epoch, max length 512, 2,000 training pairs.


In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

train_ds = ds["train"].shuffle(seed=42).select(range(2000))
eval_ds = ds["test"].select(range(100))

def train_dpo(beta):
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")

    cfg = DPOConfig(
        output_dir=f"dpo-beta{beta}",
        beta=beta,
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_length=512,
        fp16=True,
        logging_steps=10,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
    )
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules="all-linear", task_type="CAUSAL_LM")

    trainer = DPOTrainer(model=model, args=cfg, train_dataset=train_ds,
                         eval_dataset=eval_ds, processing_class=tok, peft_config=lora)
    trainer.train()
    metrics = trainer.evaluate()
    print(f"\nbeta={beta} evaluation metrics:", metrics)

    merged = trainer.model.merge_and_unload().to(torch.float16)
    repo = f"{HF_USER}/qwen2.5-0.5b-dpo-beta{str(beta).replace('.', '')}"
    merged.push_to_hub(repo)
    tok.push_to_hub(repo)
    log = trainer.state.log_history

    del trainer, model, merged
    gc.collect(); torch.cuda.empty_cache()
    return repo, metrics, log

repo_01, metrics_01, log_01 = train_dpo(0.1)   # Experiment A: baseline run
repo_05, metrics_05, log_05 = train_dpo(0.5)   # Experiment B: additional beta
print("Models uploaded:", repo_01, "and", repo_05)


### Step 5b: Save and plot the training logs
**Lab deliverable:** DPO training output and loss logs. Take a screenshot of the plots for your report.
A healthy run starts with a loss near 0.69 and goes down, while reward accuracy and margin go up.

In [ ]:
import matplotlib.pyplot as plt

pd.DataFrame(log_01).to_csv("logs_beta0.1.csv", index=False)
pd.DataFrame(log_05).to_csv("logs_beta0.5.csv", index=False)

def train_rows(log):
    return pd.DataFrame([l for l in log if "loss" in l])

for col in ["loss", "rewards/accuracies", "rewards/margins"]:
    plt.figure(figsize=(6.5, 3.6))
    for name, log in [("beta = 0.1", log_01), ("beta = 0.5", log_05)]:
        d = train_rows(log)
        if col in d.columns:
            plt.plot(d["step"], d[col], marker="o", label=name)
    plt.title(col); plt.xlabel("training step"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig("plot_" + col.replace("/", "_") + ".png", dpi=150, bbox_inches="tight")
    plt.show()

print("beta 0.1 final metrics:", {k: round(v, 4) for k, v in metrics_01.items() if isinstance(v, float)})
print("beta 0.5 final metrics:", {k: round(v, 4) for k, v in metrics_05.items() if isinstance(v, float)})


### Step 5c: Check both models exist on Hugging Face and add a model card
This also writes a model card (README) into each repo automatically.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for beta, repo in [("0.1", repo_01), ("0.5", repo_05)]:
    print(repo, "->", api.list_repo_files(repo))
    card = f"""---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B
datasets:
- trl-lib/ultrafeedback_binarized
library_name: transformers
tags:
- dpo
- alignment
---

# Qwen2.5-0.5B aligned with DPO (beta = {beta})

Fine-tuned for LLM Lab 10 (University of Gwadar).

- Base model: Qwen/Qwen2.5-0.5B
- Method: DPO with LoRA (r = 16, alpha = 32), merged into the base weights
- Data: 2,000 pairs from trl-lib/ultrafeedback_binarized
- Training: learning rate 5e-5, batch size 2 with gradient accumulation 8, 1 epoch, max length 512, beta = {beta}
"""
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
    print("Model card uploaded: https://huggingface.co/" + repo + "\n")


## STEP 6: Run the same prompts on all three models
**Lab requirements covered:** test the aligned model with the same prompts, compare base and aligned responses, compare the two beta values.

The base answers here are identical to Step 3, because the settings are the same and decoding is deterministic.

In [ ]:
models = {"base": BASE, "dpo_beta_0.1": repo_01, "dpo_beta_0.5": repo_05}

rows = {"task_type": task_types, "prompt": prompts}
for name, model_id in models.items():
    tok, m = load_model(model_id)
    rows[name] = [generate(tok, m, p) for p in prompts]
    del m
    gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows)
for name in models:
    df[f"{name}_score"] = ""      # you fill these with a number from 1 to 5
df["winner"] = ""                 # you fill: Base, DPO or Tie
df["evidence"] = ""               # you fill: one line explaining why
df.to_csv("evaluation.csv", index=False)

for i, p in enumerate(prompts):
    print(f"\n########## PROMPT {i + 1} [{task_types[i]}] ##########\n{p}")
    for name in models:
        print(f"\n--- {name} ---\n{df.loc[i, name]}")


## STEP 7: Score the answers (YOU DO THIS)
**Lab requirement:** a simple, consistent evaluation. Do not assume the aligned model is better. Let your evidence decide.

**Method:** score every answer from **1 to 5**, judging four things together: helpfulness, relevance, clarity and instruction following. Use the same rule for all three models.

| Score | Meaning |
|---|---|
| 5 | Helpful, on topic, clear and does exactly what was asked |
| 3 | Partly useful, with mistakes, rambling or ignoring part of the request |
| 1 | Useless, off topic, repeated text or broken output |

**How to do it:**
1. In the left sidebar of Colab open the **Files** icon and download `evaluation.csv`.
2. Open it in Excel. Fill in the three `_score` columns (1 to 5), then `winner` (Base, DPO or Tie) and one line of `evidence` for each prompt.
3. Save the file and run the next cell to upload it. The cell computes your averages.

In [ ]:
from google.colab import files

uploaded = files.upload()          # choose your filled evaluation.csv
df = pd.read_csv(next(iter(uploaded)))

score_cols = ["base_score", "dpo_beta_0.1_score", "dpo_beta_0.5_score"]
for c in score_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

summary = df[score_cols].mean().round(2)
summary.index = ["Base", "DPO beta=0.1", "DPO beta=0.5"]
print("AVERAGE SCORE (1-5):\n", summary, "\n")

print("AVERAGE BY TASK TYPE:")
print(df.groupby("task_type")[score_cols].mean().round(2).to_string(), "\n")

print("WINNER COUNTS:")
print(df["winner"].value_counts().to_string())

summary.to_csv("summary_scores.csv")
df.to_csv("evaluation_scored.csv", index=False)


## STEP 8: Final analysis (YOU DO THIS)
Write your answers in a text cell below, using your scores, the loss plots and the actual answers as evidence:

1. Did DPO improve the model overall? Explain using your results.
2. Which types of prompts improved the most?
3. Were there any prompts where the base model performed better?
4. What characteristics did you notice in the chosen responses of the preference dataset?
5. How did changing beta (0.1 versus 0.5) affect the model?
6. What do you think DPO actually changed in the model's behaviour?

Hint for question 5: compare the answers, the final loss and the reward margin of the two runs. A higher beta keeps the model closer to the base model.


## STEP 9: Launch the two apps
**Teacher's request:** App 1 = base model (before DPO), App 2 = DPO model (after DPO).

The cell prints **two public links**. Open each in a new tab, click your example prompts and take screenshots for the report.
The cell keeps running so that the links stay alive. Stop it only when you finish the demo. The links stop working when Colab stops.

In [ ]:
import gradio as gr, time

tok_b, base_model = load_model(BASE)
tok_1, dpo_01 = load_model(repo_01)
tok_5, dpo_05 = load_model(repo_05)

def app1_fn(prompt, max_tokens):
    return generate(tok_b, base_model, prompt, int(max_tokens))

def app2_fn(prompt, beta, max_tokens):
    tok, m = (tok_1, dpo_01) if beta == "0.1" else (tok_5, dpo_05)
    return generate(tok, m, prompt, int(max_tokens))

app1 = gr.Interface(
    fn=app1_fn,
    inputs=[gr.Textbox(label="Your prompt", lines=3),
            gr.Slider(50, 300, value=150, step=10, label="Max new tokens")],
    outputs=gr.Textbox(label="BASE model response (before DPO)", lines=12),
    title="App 1 - Base Model (before DPO)",
    description="Qwen2.5-0.5B before alignment.",
    examples=[[p, 150] for p in prompts],
)

app2 = gr.Interface(
    fn=app2_fn,
    inputs=[gr.Textbox(label="Your prompt", lines=3),
            gr.Radio(["0.1", "0.5"], value="0.1", label="DPO beta value"),
            gr.Slider(50, 300, value=150, step=10, label="Max new tokens")],
    outputs=gr.Textbox(label="DPO model response (after alignment)", lines=12),
    title="App 2 - DPO Model (after DPO)",
    description="Qwen2.5-0.5B aligned with DPO. Choose beta 0.1 or 0.5.",
    examples=[[p, "0.1", 150] for p in prompts],
)

_, _, url1 = app1.launch(share=True, prevent_thread_lock=True, server_port=7860)
_, _, url2 = app2.launch(share=True, prevent_thread_lock=True, server_port=7861)

print("\n==============================================")
print("App 1 (before DPO):", url1)
print("App 2 (after DPO) :", url2)
print("==============================================")
print("Copy both links into your Vercel page. Keep this cell running.")

while True:
    time.sleep(60)
